# 02 - LoRA Fine-Tuning Tutorial (distilgpt2)

This notebook focuses on the LoRA branch and inspects real training/evaluation artifacts produced by the run.

## LoRA training metrics (from artifacts)

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent.resolve()
ART = ROOT / 'artifacts'
MODELS = ROOT / 'models'

lora_train = json.loads((ART / 'metrics' / 'lora_train_metrics.json').read_text())
lora_train

{'train': {'train_runtime': 6.9273,
  'train_samples_per_second': 69.291,
  'train_steps_per_second': 4.331,
  'total_flos': 9199127494656.0,
  'train_loss': 4.146671994527181,
  'epoch': 0.26666666666666666},
 'eval': {'eval_loss': 3.566974401473999,
  'eval_runtime': 0.7485,
  'eval_samples_per_second': 400.813,
  'eval_steps_per_second': 50.77,
  'epoch': 0.26666666666666666},
 'model': 'distilgpt2',
 'method': 'lora',
 'train_samples': 1800,
 'validation_samples': 300}

## Compare LoRA baseline vs tuned performance

In [2]:
metrics = pd.read_csv(ART / 'metrics' / 'evaluation_metrics.csv')
metrics[metrics['run_name'].isin(['baseline_lora','tuned_lora'])]

,run_name,model_name,quantized_4bit,adapter_path,n_samples,accuracy,macro_f1
0,baseline_lora,distilgpt2,False,NaN,40,0.0000,0.000000
1,tuned_lora,distilgpt2,False,/home/ahmad/AI/Projects/lora-qlora-finetuning-...,80,0.3125,0.098384


## Inspect tuned LoRA predictions

In [3]:
preds = pd.read_csv(ART / 'tables' / 'predictions.csv')
preds[preds['run_name']=='tuned_lora'].head(20)

,run_name,text,gold_label,raw_completion,pred_label
40,tuned_lora,i was feeling really troubled and down over wh...,sadness,"sadness, joy,",sadness
41,tuned_lora,i feel so thrilled to have three such distingu...,joy,"sadness, joy,",sadness
42,tuned_lora,i feel is that the most likeable characters ar...,joy,"sadness, joy,",sadness
43,tuned_lora,i tune out the rest of the world and focus on ...,joy,"love, joy,",joy
44,tuned_lora,i sit here writing this i feel unhappy inside,sadness,"sadness, joy,",sadness
45,tuned_lora,im feeling and if ive liked being pregnant,love,"sadness, joy,",sadness
46,tuned_lora,im very hurt and i feel unimportant,sadness,"sadness, joy,",sadness
47,tuned_lora,i used to be able to hang around talk with the...,anger,"sadness, joy,",sadness
48,tuned_lora,i don t have the feeling of divine vibrations,joy,"sadness, joy,",sadness
49,tuned_lora,i vented my feelings towards the pathetic excu...,sadness,"sadness, joy,",sadness


## LoRA adapter path and files

In [4]:
lora_adapter = MODELS / 'lora_distilgpt2' / 'adapter'
print(lora_adapter)
for p in sorted(lora_adapter.glob('*')):
    print(p.name)

/home/ahmad/AI/Projects/lora-qlora-finetuning-lab/models/lora_distilgpt2/adapter
README.md
adapter_config.json
adapter_model.safetensors
tokenizer.json
tokenizer_config.json
training_args.bin
